In [2]:
import numpy as np

# ==================== CNN using TensorFlow ====================
print("="*60)
print("CNN Implementation using TensorFlow")
print("="*60)

import tensorflow as tf
from tensorflow.keras import layers, models

# Load dataset
print("\nLoading MNIST dataset...")
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train, x_test = x_train / 255.0, x_test / 255.0

# Expand dimensions (add channel dimension)
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Build CNN model
print("Building TensorFlow CNN model...")
model_tf = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), name='conv1'),
    layers.MaxPooling2D((2, 2), name='pool1'),
    layers.Conv2D(64, (3, 3), activation='relu', name='conv2'),
    layers.MaxPooling2D((2, 2), name='pool2'),
    layers.Flatten(name='flatten'),
    layers.Dense(64, activation='relu', name='fc1'),
    layers.Dense(10, activation='softmax', name='output')
])

# Compile
model_tf.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Display model architecture
print("\nTensorFlow Model Architecture:")
model_tf.summary()

# Train
print("\nTraining TensorFlow CNN (3 epochs)...")
history_tf = model_tf.fit(x_train, y_train, epochs=3, batch_size=128, verbose=1)

# Evaluate
print("\nEvaluating TensorFlow CNN on test data...")
test_loss_tf, test_accuracy_tf = model_tf.evaluate(x_test, y_test, verbose=0)
print(f"TensorFlow CNN - Test Loss: {test_loss_tf:.4f}, Test Accuracy: {test_accuracy_tf:.4f}")


# ==================== CNN using PyTorch ====================
print("\n" + "="*60)
print("CNN Implementation using PyTorch")
print("="*60)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

# Define CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        # Conv layer 1 + Pool
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)  # 28x28 -> 14x14

        # Conv layer 2 + Pool
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)  # 14x14 -> 7x7

        # Flatten
        x = x.view(x.size(0), -1)  # 64*7*7

        # Fully connected layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Initialize model, loss, and optimizer
print("\nBuilding PyTorch CNN model...")
model_pytorch = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pytorch.parameters(), lr=0.001)

# Display model architecture
print("\nPyTorch Model Architecture:")
print(model_pytorch)

# Prepare data
print("\nPreparing data...")
# Convert from (batch, height, width, channels) to (batch, channels, height, width)
x_train_pytorch = np.transpose(x_train, (0, 3, 1, 2))
x_test_pytorch = np.transpose(x_test, (0, 3, 1, 2))

x_train_tensor = torch.from_numpy(x_train_pytorch).float().to(device)
y_train_tensor = torch.from_numpy(y_train).long().to(device)
x_test_tensor = torch.from_numpy(x_test_pytorch).float().to(device)
y_test_tensor = torch.from_numpy(y_test).long().to(device)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Training loop
print("Training PyTorch CNN (2 epochs)...")
num_epochs = 2
for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model_pytorch(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluate on test data
print("\nEvaluating PyTorch CNN on test data...")
model_pytorch.eval()
with torch.no_grad():
    test_outputs = model_pytorch(x_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy_pytorch = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f"PyTorch CNN - Test Accuracy: {accuracy_pytorch:.4f}")

# Summary
print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"TensorFlow CNN Test Accuracy: {test_accuracy_tf:.4f}")
print(f"PyTorch CNN Test Accuracy: {accuracy_pytorch:.4f}")
print("="*60)


CNN Implementation using TensorFlow

Loading MNIST dataset...
Building TensorFlow CNN model...

TensorFlow Model Architecture:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)


Training TensorFlow CNN (3 epochs)...
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9314 - loss: 0.2348
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9805 - loss: 0.0622
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9861 - loss: 0.0449

Evaluating TensorFlow CNN on test data...
TensorFlow CNN - Test Loss: 0.0382, Test Accuracy: 0.9874

CNN Implementation using PyTorch

Using device: cuda

Building PyTorch CNN model...

PyTorch Model Architecture:
CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)

Preparing data...
Training PyTorch CNN (2 epochs)...
Epoch [1/2], Loss: 0.2801
Epoch [2/2], Loss: 0.0710



prac13


In [1]:
import numpy as np

print("="*70)
print("MNIST Handwritten Character Detection")
print("Comparison: TensorFlow, Keras, and PyTorch")
print("="*70)

# ==================== TensorFlow Implementation ====================
print("\n" + "="*70)
print("1. TensorFlow Implementation")
print("="*70)

import tensorflow as tf
from tensorflow.keras import layers, models

print("\nLoading MNIST dataset...")
mnist = tf.keras.datasets.mnist
(x_train_tf, y_train_tf), (x_test_tf, y_test_tf) = mnist.load_data()

print(f"Dataset shape - Training: {x_train_tf.shape}, Test: {x_test_tf.shape}")

# Normalize
x_train_tf = x_train_tf / 255.0
x_test_tf = x_test_tf / 255.0

# Build TensorFlow model
print("\nBuilding TensorFlow model...")
model_tf = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu', name='hidden'),
    layers.Dense(10, activation='softmax', name='output')
])

model_tf.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print("\nTensorFlow Model Summary:")
model_tf.summary()

print("\nTraining TensorFlow model (3 epochs)...")
history_tf = model_tf.fit(x_train_tf, y_train_tf, epochs=3, batch_size=128, verbose=1)

print("\nEvaluating TensorFlow model...")
loss_tf, accuracy_tf = model_tf.evaluate(x_test_tf, y_test_tf, verbose=0)
print(f"TensorFlow - Test Loss: {loss_tf:.4f}, Test Accuracy: {accuracy_tf:.4f}")


# ==================== Keras Implementation ====================
print("\n" + "="*70)
print("2. Keras (Integrated with TensorFlow) Implementation")
print("="*70)

from tensorflow import keras
from tensorflow.keras import layers

print("\nLoading MNIST dataset...")
(x_train_keras, y_train_keras), (x_test_keras, y_test_keras) = keras.datasets.mnist.load_data()

# Normalize
x_train_keras = x_train_keras / 255.0
x_test_keras = x_test_keras / 255.0

# Build Keras model
print("Building Keras model...")
model_keras = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu', name='hidden'),
    layers.Dense(10, activation='softmax', name='output')
])

model_keras.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

print("\nKeras Model Summary:")
model_keras.summary()

print("\nTraining Keras model (3 epochs)...")
history_keras = model_keras.fit(x_train_keras, y_train_keras, epochs=3, batch_size=128, verbose=1)

print("\nEvaluating Keras model...")
loss_keras, accuracy_keras = model_keras.evaluate(x_test_keras, y_test_keras, verbose=0)
print(f"Keras - Test Loss: {loss_keras:.4f}, Test Accuracy: {accuracy_keras:.4f}")


# ==================== PyTorch Implementation ====================
print("\n" + "="*70)
print("3. PyTorch Implementation")
print("="*70)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

print("\nLoading MNIST dataset...")
mnist_torch = keras.datasets.mnist
(x_train_pt, y_train_pt), (x_test_pt, y_test_pt) = mnist_torch.load_data()

# Normalize
x_train_pt = x_train_pt / 255.0
x_test_pt = x_test_pt / 255.0

# Define PyTorch model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.hidden = nn.Linear(784, 128)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x

# Initialize model, loss, and optimizer
print("\nBuilding PyTorch model...")
model_pt = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

print("\nPyTorch Model Architecture:")
print(model_pt)

# Prepare data loaders
print("\nPreparing PyTorch data loaders...")
x_train_pt_tensor = torch.from_numpy(x_train_pt).float().to(device)
y_train_pt_tensor = torch.from_numpy(y_train_pt).long().to(device)
x_test_pt_tensor = torch.from_numpy(x_test_pt).float().to(device)
y_test_pt_tensor = torch.from_numpy(y_test_pt).long().to(device)

train_dataset = TensorDataset(x_train_pt_tensor, y_train_pt_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataset = TensorDataset(x_test_pt_tensor, y_test_pt_tensor)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Training loop
print("\nTraining PyTorch model (3 epochs)...")
num_epochs = 3
for epoch in range(num_epochs):
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_pt(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}, Accuracy: {accuracy:.2f}%")

# Evaluate on test data
print("\nEvaluating PyTorch model...")
model_pt.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model_pt(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_pt = 100 * correct / total
print(f"PyTorch - Test Accuracy: {accuracy_pt:.2f}%")


# ==================== Comparison and Summary ====================
print("\n" + "="*70)
print("Performance Comparison Summary")
print("="*70)
print(f"\nTensorFlow  - Test Accuracy: {accuracy_tf*100:.2f}%")
print(f"Keras       - Test Accuracy: {accuracy_keras*100:.2f}%")
print(f"PyTorch     - Test Accuracy: {accuracy_pt:.2f}%")
print("\n" + "="*70)
print("All three frameworks successfully trained on MNIST dataset!")
print("Framework Characteristics:")
print("  • TensorFlow: High-level API, easy to use, production-ready")
print("  • Keras: User-friendly, integrated with TensorFlow")
print("  • PyTorch: Flexible, dynamic computation graphs, research-friendly")
print("="*70)


MNIST Handwritten Character Detection
Comparison: TensorFlow, Keras, and PyTorch

1. TensorFlow Implementation

Loading MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset shape - Training: (60000, 28, 28), Test: (10000, 28, 28)

Building TensorFlow model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



TensorFlow Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)


Training Keras model (3 epochs)...
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8985 - loss: 0.3645
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9513 - loss: 0.1700
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9658 - loss: 0.1189

Evaluating Keras model...
Keras - Test Loss: 0.1097, Test Accuracy: 0.9659

3. PyTorch Implementation

Using device: cuda

Loading MNIST dataset...

Building PyTorch model...

PyTorch Model Architecture:
Net(
  (hidden): Linear(in_features=784, out_features=128, bias=True)
  (output): Linear(in_features=128, out_features=10, bias=True)
)

Preparing PyTorch data loaders...

Training PyTorch model (3 epochs)...
Epoch [1/3], Loss: 0.4126, Accuracy: 89.39%
Epoch [2/3], Loss: 0.1925, Accuracy: 94.54%
Epoch [3/3], Loss: 0.1402, Accuracy: 96.08%

Evaluating PyTorch model...
PyTorch - Test Accuracy: 96.25%

Performance Comparison Summary

TensorFlow  - Test Accuracy: 96.64%
Keras       - Test Accuracy: 96.5

In [ ]:
pip install nbconvert

In [3]:
import os

# Get the current notebook filename
notebook_name = "CNN_Implementations_TensorFlow_PyTorch.ipynb" # Replace with the actual notebook name if different

# Convert the notebook to HTML
!jupyter nbconvert --to html "{notebook_name}"

print(f"Notebook '{notebook_name}' converted to HTML successfully!")

[NbConvertApp] WARNING | pattern 'CNN_Implementations_TensorFlow_PyTorch.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.ans